In [3]:
# Requirements: geopandas, shapely, fiona
# If you run in a new environment:
# pip install geopandas fiona shapely pyproj rtree

import geopandas as gpd
import fiona
from pathlib import Path
from collections import defaultdict
from shapely.ops import unary_union
from shapely.strtree import STRtree

# --- INPUTS ---
input_gpkg = r"P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Tunnels.gpkg"
input_layer = None  # e.g. "tunnels". If None, use the first layer.
output_gpkg = str(Path(input_gpkg).with_name("Tunnels_dissolved.gpkg"))
output_layer = "tunnels_dissolved"

# --- Read layer ---
if not Path(input_gpkg).exists():
    raise FileNotFoundError(f"Input not found: {input_gpkg}")

try:
    layers = fiona.listlayers(input_gpkg)
except Exception as e:
    raise RuntimeError(f"Could not list layers in {input_gpkg}: {e}")

layer_to_read = input_layer or layers[0]
print(f"Reading layer: {layer_to_read}")
gdf = gpd.read_file(input_gpkg, layer=layer_to_read)

# Drop empty/null geometries
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].reset_index(drop=True)
if len(gdf) == 0:
    raise ValueError("No geometries to process after filtering.")

# --- Build spatial index & connected components based on intersects (includes touches) ---
geoms = list(gdf.geometry)
tree = STRtree(geoms)

# Map WKB -> indices for Shapely 1.x fallback (query returns geometry objects)
wkb_to_indices = defaultdict(list)
for idx, geom in enumerate(geoms):
    wkb_to_indices[geom.wkb].append(idx)

# Build adjacency robust across Shapely 1.x and 2.x
adj = {i: set() for i in range(len(geoms))}
for i, gi in enumerate(geoms):
    try:
        # Shapely 2.x: returns array of indices; predicate pre-filters intersections
        cand_idx = [int(x) for x in tree.query(gi, predicate="intersects")]
    except TypeError:
        # Shapely 1.x: returns geometry objects; filter intersections manually
        cand_geoms = tree.query(gi)  # bbox candidates
        cand_idx = []
        for gj in cand_geoms:
            cand_idx.extend(wkb_to_indices[gj.wkb])
        cand_idx = [j for j in set(cand_idx) if j != i and gi.intersects(geoms[j])]

    for j in cand_idx:
        if i == j:
            continue
        adj[i].add(j)
        adj[j].add(i)

# --- Connected components via DFS ---
visited = set()
components = []
for i in range(len(geoms)):
    if i in visited:
        continue
    stack = [i]
    comp = []
    while stack:
        k = stack.pop()
        if k in visited:
            continue
        visited.add(k)
        comp.append(k)
        stack.extend([n for n in adj[k] if n not in visited])
    components.append(comp)

print(f"Found {len(components)} connected components.")

# --- Dissolve each component ---
dissolved_geoms = []
for comp in components:
    union_geom = unary_union([geoms[idx] for idx in comp])
    dissolved_geoms.append(union_geom)

# --- Output ---
out = gpd.GeoDataFrame({"component_id": range(len(dissolved_geoms))},
                       geometry=dissolved_geoms, crs=gdf.crs)

# Optional: fix tiny artifacts
# out["geometry"] = out["geometry"].buffer(0)

print(f"Writing {len(out)} dissolved features to {output_gpkg} (layer={output_layer})")
out.to_file(output_gpkg, layer=output_layer, driver="GPKG")
print("Done.")

Reading layer: Tunnels
Found 59 connected components.
Writing 59 dissolved features to P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Tunnels_dissolved.gpkg (layer=tunnels_dissolved)
Done.


In [4]:
# --- Paths ---
tunnels_gpkg = r"P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Tunnels_dissolved.gpkg"
tunnels_layer = "tunnels_dissolved"  # per earlier output
roads_parquet = r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Filtering_all_columns.parquet"

# --- Load tunnels as GeoDataFrame ---
tunnels = gpd.read_file(tunnels_gpkg, layer=tunnels_layer)
tunnels = tunnels[tunnels.geometry.notna() & ~tunnels.geometry.is_empty].reset_index(drop=True)

# --- Load roads parquet ---
# If parquet has WKT column instead of native geometry, adapt accordingly.
roads = gpd.read_parquet(roads_parquet)
if roads.crs is None:
    # Try to inherit CRS from tunnels if roads have same coordinates. Adjust if you know the correct CRS.
    roads = roads.set_crs(tunnels.crs, allow_override=True)

# Ensure both are in same CRS
if tunnels.crs != roads.crs:
    roads = roads.to_crs(tunnels.crs)

# --- Spatial overlay: roads onto tunnels ---
# Keep roads that intersect tunnels and attach road attributes to tunnels
joined = gpd.sjoin(tunnels, roads, how="left", predicate="intersects")

# --- Remove tunnels with no overlay (no road match) ---
# After sjoin, unmatched rows have NaNs for road columns; identify via sjoin index column
# sjoin creates 'index_right' by default for right side index
has_overlay = joined["index_right"].notna()
tunnels_overlaid = joined[has_overlay].copy()

# --- Add flooded column based on overlaid road row 'flooded_tunnel_entrance' ---
# Assume the road parquet has a column named 'flooded_tunnel_entrance' indicating 1/0.
# Coerce to integer 0/1 safely.
if "flooded_tunnel_entrance" in tunnels_overlaid.columns:
    tunnels_overlaid["flooded"] = (tunnels_overlaid["flooded_tunnel_entrance"].fillna(0).astype(int) == 1).astype(int)
else:
    # If the column is missing, set flooded to 0 and warn
    print("Warning: 'flooded_tunnel_entrance' not found in roads; setting flooded=0.")
    tunnels_overlaid["flooded"] = 0

# --- Optional: if multiple roads hit the same tunnel, aggregate to one tunnel row ---
# Keep one record per original tunnel by grouping on original index (or component_id) and taking max flooded.
group_key = "component_id" if "component_id" in tunnels_overlaid.columns else tunnels.index.name or "index"
if group_key not in tunnels_overlaid.columns:
    tunnels_overlaid[group_key] = tunnels_overlaid.index

agg = (
    tunnels_overlaid
    .groupby(group_key, as_index=False)
    .agg({
        "geometry": "first",     # tunnels geometry is the same across joined duplicates
        "flooded": "max"         # any hit with flooded_tunnel_entrance=1 => flooded=1
    })
)
agg = gpd.GeoDataFrame(agg, geometry="geometry", crs=tunnels.crs)

# --- Save updated tunnels back to GeoPackage (overwriting or new layer) ---
output_layer_updated = "tunnels_dissolved_with_flooded"
print(f"Writing {len(agg)} tunnels with flooded flag to {tunnels_gpkg} (layer={output_layer_updated})")
agg.to_file(tunnels_gpkg, layer=output_layer_updated, driver="GPKG")
print("Done.")

Writing 55 tunnels with flooded flag to P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Tunnels_dissolved.gpkg (layer=tunnels_dissolved_with_flooded)


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('tunnels_dissolved_with_flooded')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('tunnels_dissolved_with_flooded')) failed: disk I/O error"


Done.
